In [4]:
# =====================================================================
# LAB 3: CHATBOT DE E-COMMERCE COM DECISION TREE
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier


# =====================================================================
# DADOS DE TREINAMENTO
# =====================================================================

dados_treino = [
    ("Onde esta meu pedido e qual o rastreio", "rastrear_pedido"),
    ("Quero saber a localizacao da minha encomenda", "rastrear_pedido"),
    ("Quero rastrear meu pedido", "rastrear_pedido"),
    ("Onde esta minha encomenda", "rastrear_pedido"),
    ("Preciso saber onde esta meu pedido", "rastrear_pedido"),

    ("Quero cancelar minha compra e pedir reembolso", "cancelar_compra"),
    ("Preciso cancelar o pedido feito ontem", "cancelar_compra"),
    ("Quero cancelar minha compra", "cancelar_compra"),
    ("Desejo cancelar meu pedido", "cancelar_compra"),
    ("Quero pedir reembolso da compra", "cancelar_compra"),

    ("Qual o horario de atendimento da loja fisica", "fora_escopo"),
    ("Qual a previsão do tempo", "fora_escopo"),
    ("Me conte uma piada", "fora_escopo"),
    ("Quem ganhou o jogo ontem", "fora_escopo"),
    ("Qual a capital do Brasil", "fora_escopo")
]


stopwords = [
    "a","o","as","os","um","uma","de","da","do","das","dos","para","por","me","meu","minha","na","no","nas","nos","e","é"
]


# =====================================================================
# 1. PRÉ-PROCESSAMENTO
# =====================================================================

def pré_processar(texto):

    # Coloca o texto em minúsculo
    # e remove pontuações
    texto_limpo = re.sub(
        r'[^\w\s]',
        '',
        texto.lower()
    ).strip()


    # Separa a frase em palavras
    tokens = texto_limpo.split()


    # TODO 3.1
    # Remove as stopwords
    tokens_filtrados = [
        palavra
        for palavra in tokens
        if palavra not in stopwords
    ]


    # Junta as palavras novamente
    return " ".join(tokens_filtrados)


# =====================================================================
# PREPARAÇÃO DOS DADOS
# =====================================================================

X_treino_limpo = [
    pré_processar(item[0])
    for item in dados_treino
]


y_treino = [
    item[1]
    for item in dados_treino
]


# =====================================================================
# 2. VETORIZAÇÃO E TREINAMENTO
# =====================================================================

vectorizer = TfidfVectorizer()


# Aprende o vocabulário e transforma
# os textos em números
X_vetorizado = vectorizer.fit_transform(
    X_treino_limpo
)


# TODO 3.2
# Criando o modelo de Árvore de Decisão
modelo_tree = DecisionTreeClassifier(
    random_state=42
)


# TODO 3.3
# Treinando o modelo
modelo_tree.fit(
    X_vetorizado,
    y_treino
)


# =====================================================================
# 3. EXTRAÇÃO DE ENTIDADE
# CÓDIGO DE RASTREIO
# =====================================================================

def extrair_codigo_rastreio(texto):

    # Procura códigos como:
    #
    # BR123456789
    # BR987654321

    match = re.search(
        r'BR\d{9}',
        texto,
        re.IGNORECASE
    )


    # TODO 3.4
    # Se encontrou o código
    if match:

        return match.group(0).upper()


    # Se não encontrou
    return None


# =====================================================================
# 4. PIPELINE NLU
# =====================================================================

def processar_ecommerce(
    mensagem_usuario,
    threshold=0.50
):

    # ---------------------------------------------------------
    # PASSO 1 - PRÉ-PROCESSAMENTO
    # ---------------------------------------------------------

    texto_p = pré_processar(
        mensagem_usuario
    )


    # ---------------------------------------------------------
    # PASSO 2 - VETORIZAÇÃO
    # ---------------------------------------------------------

    vetor_input = vectorizer.transform(
        [texto_p]
    )


    # ---------------------------------------------------------
    # PASSO 3 - PROBABILIDADES
    # ---------------------------------------------------------

    probas = modelo_tree.predict_proba(
        vetor_input
    )[0]


    # ---------------------------------------------------------
    # PASSO 4 - MAIOR CONFIANÇA
    # ---------------------------------------------------------

    maior_confianca = max(
        probas
    )


    # ---------------------------------------------------------
    # PASSO 5 - INTENÇÃO PREVISTA
    # ---------------------------------------------------------

    intencao_prevista = modelo_tree.classes_[
        probas.argmax()
    ]


    # ---------------------------------------------------------
    # PASSO 6 - EXTRAIR CÓDIGO DE RASTREIO
    # ---------------------------------------------------------

    codigo_rastreio = extrair_codigo_rastreio(
        mensagem_usuario
    )


    # ---------------------------------------------------------
    # TODO 3.5
    # VERIFICAÇÃO DE SEGURANÇA
    # ---------------------------------------------------------

    if (
        maior_confianca < threshold
        or
        intencao_prevista == "fora_escopo"
    ):

        return {
            "status": "FALLBACK",
            "intencao": intencao_prevista,
            "confianca": maior_confianca,
            "codigo_rastreio": codigo_rastreio
        }

    else:

        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": maior_confianca,
            "codigo_rastreio": codigo_rastreio
        }


# =====================================================================
# TESTE 3
# =====================================================================

if __name__ == "__main__":

    print(
        processar_ecommerce(
            "Quero rastrear o pedido BR987654321"
        )
    )

{'status': 'SUCESSO', 'intencao': np.str_('rastrear_pedido'), 'confianca': np.float64(1.0), 'codigo_rastreio': 'BR987654321'}
